# Capstone, a retry loop that terminates

**Scenario:** an agent turns investigator narratives into adverse event records for a trial safety
database. A record fails validation. The loop asks again, gets the same answer, fails again. Someone
caps it at three tries and ships. Then it starts succeeding, by inventing the value that blocked it.

A retry with no diagnosis is a rejected form handed back blank. This capstone builds the other kind,
**a form handed back with the wrong box circled**, and a shelf for the forms no answer can fix.

## Mechanics

Six settings decide whether a repair loop is safe. Most builds carry only the first.

| Setting | Value here | Why it exists |
|---|---|---|
| `max_attempts` | a small integer | the only thing that makes the loop finite |
| what is fed back | `ValidationError.errors()`, as `loc` and `msg` | a diagnosis the model can act on |
| repairable | the reply was wrong, the narrative was not | asking again can work |
| poison | the narrative and the rule disagree | no reply can fix it, so nothing is sent back |
| dead letter record | the raw payload, the reasons, the attempt count | what a person needs to work it |
| `Agent(..., retries=n)` | pydantic-ai | raises `UnexpectedModelBehavior` once the budget is spent |

The poison row is the one people skip. A loop with feedback and no poison path does not hang. It
lies.

## The picture

![A bounded loop with one exit for repairs and one for the shelf](images/retry-that-terminates.svg)

Two exits, and both are reachable from every attempt. A loop with only the happy exit is the loop
that invents a value to reach it.

## The cost

```
worst case per record = max_attempts x cost of one call
```

That bound is the cheap half. The expensive half is a record that passed because a date moved. Before
or after the first dose decides whether the event is counted at all.

## The failure

Two narratives, and the shape the model is asked for. The protocol rules stay out of the prompt,
because a rule in a prompt is a request.

In [1]:
import json
from vault import get_client, load_env, model_for

load_env()
client = get_client("09-programmatic-guardrails/03-capstone-a-retry-loop-that-terminates")

SYSTEM = "You turn investigator narratives into adverse event records as JSON."
FIXABLE = ("Site 03. Subject 7741. On study day 12 the participant developed grade 4 "
           "neutropenia requiring hospital admission. Related to study drug.")
POISON = ("Site 03. Subject S-03-9002. The participant reported grade 2 nausea beginning "
          "two days before the first dose of study drug. Considered unrelated.")

FIELDS = {"subject_id": {"type": "string"}, "term": {"type": "string"},
          "grade": {"type": "integer"}, "onset_day": {"type": "integer"},
          "serious": {"type": "boolean"}}
FORMAT = {"type": "json_schema", "json_schema": {
    "name": "adverse_event", "strict": True,
    "schema": {"type": "object", "properties": FIELDS, "required": list(FIELDS),
               "additionalProperties": False}}}

One call. The `feedback` argument is the whole subject of this notebook, and it starts out unused.

In [2]:
def extract_once(narrative, feedback=None, previous=None):
    """One extraction. A repair turn carries the rejected record and the reasons."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": narrative}]
    if previous:
        messages.append({"role": "assistant", "content": json.dumps(previous)})
    if feedback:
        messages.append({"role": "user", "content": feedback})
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=300, response_format=FORMAT,
        messages=messages)
    return json.loads(reply.choices[0].message.content)

The protocol, as types. Day one is the first dose, so anything earlier is not a treatment emergent
event and cannot be filed as one.

In [3]:
from pydantic import BaseModel, Field, ValidationError, model_validator


class AdverseEvent(BaseModel):
    """One safety record. The protocol rules live here, never in the prompt."""

    subject_id: str = Field(pattern=r"^S-\d{2}-\d{4}$")
    term: str = Field(min_length=3)
    grade: int = Field(ge=1, le=5)
    onset_day: int = Field(ge=1)
    serious: bool

    @model_validator(mode="after")
    def high_grade_is_serious(self):
        if self.grade >= 4 and not self.serious:
            raise ValueError("grade 4 or 5 must be recorded as serious")
        return self

Now the loop almost everyone writes first. It retries, it is bounded, and it learns nothing between
attempts.

In [4]:
seen = []
for attempt in range(3):
    raw = extract_once(FIXABLE)                 # same prompt, no diagnosis
    try:
        AdverseEvent(**raw)
        seen.append("ok")
    except ValidationError as exc:
        seen.append(exc.errors()[0]["type"])

print(f"subject_id returned : {raw['subject_id']!r}")
print(f"three attempts      : {seen}")
assert "ok" in seen, f"three identical attempts, three identical failures: {seen}"

subject_id returned : '7741'
three attempts      : ['string_pattern_mismatch', 'string_pattern_mismatch', 'string_pattern_mismatch']


AssertionError: three identical attempts, three identical failures: ['string_pattern_mismatch', 'string_pattern_mismatch', 'string_pattern_mismatch']

## The diagnosis

The same failure three times, and nothing changed between attempts. The model was never told the
record was rejected, nor which field, nor what the field must look like.

The narrative holds the site and the subject number, and the protocol wants them joined. The model can
make that repair, and made it zero times out of three, because nobody asked.

A bound made the loop finite. It did not make the later attempts worth paying for.

## The fix

Send back the validator's own words. They name the field and the rule. The repair turn carries the
rejected record too, so the model sees what it wrote.

In [5]:
def repair_message(exc):
    """The validator's own words, which is the only diagnosis worth sending."""
    lines = [f"{e['loc'][0]}: {e['msg']}" for e in exc.errors()]
    return ("That record failed validation:\n" + "\n".join(lines)
            + "\nReturn the corrected record.")

Try it by hand on both narratives first. The second one is the reason this notebook exists.

In [6]:
first = extract_once(FIXABLE)
try:
    AdverseEvent(**first)
except ValidationError as exc:
    second = extract_once(FIXABLE, repair_message(exc), previous=first)

print(f"fixable, before feedback: subject_id={first['subject_id']!r}")
print(f"fixable, after feedback : subject_id={second['subject_id']!r}")

bad = extract_once(POISON)
try:
    AdverseEvent(**bad)
except ValidationError as exc:
    fixed = extract_once(POISON, repair_message(exc), previous=bad)

print(f"\nthe narrative says the event began before the first dose")
print(f"poison, before feedback : onset_day={bad['onset_day']}")
print(f"poison, after feedback  : onset_day={fixed['onset_day']}   <- invented")

fixable, before feedback: subject_id='03-7741'
fixable, after feedback : subject_id='S-03-7741'

the narrative says the event began before the first dose
poison, before feedback : onset_day=-2
poison, after feedback  : onset_day=1   <- invented


The second record now passes every rule and is false. The loop moved an event from before the first
dose to the day of it, which decides whether it is counted. Run that repair turn four times and it
lands on day one four times, so this is the behaviour and not a bad draw.

Some errors must never be sent back. Which ones is a judgement about the source, so it belongs in
your code.

In [7]:
POISON_FIELDS = {"onset_day", "grade"}      # values a reply can only invent


def is_repairable(errors):
    """A format error is worth asking about. A value that fights the narrative is not."""
    return all(e["loc"][0] not in POISON_FIELDS for e in errors)

Now the loop. Bounded, feeding back a diagnosis, stopping the moment an error is poison, and taking
its extractor as an argument so a test can hand it one that calls nothing.

In [8]:
def extract(narrative, max_attempts=3, ask=extract_once):
    """Bounded, with a diagnosis the model can act on, and a shelf for the rest."""
    feedback, errors, raw, attempt = None, [], None, 0
    for attempt in range(1, max_attempts + 1):
        raw = ask(narrative, feedback, previous=raw)
        try:
            return {"status": "ok", "record": AdverseEvent(**raw), "attempts": attempt}
        except ValidationError as exc:
            errors = exc.errors()
            if not is_repairable(errors):
                break
            feedback = repair_message(exc)
    return {"status": "parked", "raw": raw, "attempts": attempt,
            "reasons": [f"{e['loc'][0]}: {e['msg']}" for e in errors]}

Both narratives, one function, two different endings.

In [9]:
ok = extract(FIXABLE)
parked = extract(POISON)

print(f"fixable narrative: {ok['status']} after {ok['attempts']} attempts")
print(f"  {ok['record'].subject_id}  grade {ok['record'].grade}  day {ok['record'].onset_day}")
print(f"poison narrative : {parked['status']} after {parked['attempts']} attempt")
for reason in parked["reasons"]:
    print(f"  {reason}")

print(f"\nbefore: 3 attempts, no record, no diagnosis, nothing on a shelf")
print(f"after : {ok['attempts']} attempts for one record, and the impossible one is parked")

fixable narrative: ok after 2 attempts
  S-03-7741  grade 4  day 12
poison narrative : parked after 1 attempt
  onset_day: Input should be greater than or equal to 1

before: 3 attempts, no record, no diagnosis, nothing on a shelf
after : 2 attempts for one record, and the impossible one is parked


pydantic-ai ships this loop, and its budget is a constructor argument rather than a habit. The stub
below calls no network, so this cell needs no key.

In [10]:
import nest_asyncio
from pydantic_ai import Agent, UnexpectedModelBehavior
from pydantic_ai.messages import ModelResponse, TextPart
from pydantic_ai.models.function import FunctionModel

nest_asyncio.apply()          # run_sync inside the notebook's own event loop
TRIES = {"n": 0}
WRONG = ('{"subject_id": "7741", "term": "rash", "grade": 2, '
         '"onset_day": 4, "serious": false}')


def always_wrong(messages, info):
    """A model that never satisfies the type. Offline, and deliberately hopeless."""
    TRIES["n"] += 1
    return ModelResponse(parts=[TextPart(WRONG)])


try:
    Agent(FunctionModel(always_wrong), output_type=AdverseEvent, retries=2).run_sync("extract")
except UnexpectedModelBehavior as exc:
    print(f"pydantic-ai stopped on its own: {exc}")
print(f"calls made: {TRIES['n']}")

pydantic-ai stopped on its own: Exceeded maximum output retries (2)
calls made: 3


## The gate

The property to hold is that the loop is finite whatever the model does. This test hands it an
extractor that can never succeed.

In [11]:
def test_the_loop_cannot_run_forever():
    calls = {"n": 0}

    def never_valid(narrative, feedback=None, previous=None):
        calls["n"] += 1
        return {"subject_id": "7741", "term": "rash", "grade": 2,
                "onset_day": 4, "serious": False}

    outcome = extract("any narrative", max_attempts=3, ask=never_valid)
    assert outcome["status"] == "parked", "a hopeless record was accepted"
    assert calls["n"] == 3, f"{calls['n']} calls against a budget of 3"


test_the_loop_cannot_run_forever()
print("gate holds: three attempts, then the shelf, whatever the model returns")

gate holds: three attempts, then the shelf, whatever the model returns


Change `for attempt in range(1, max_attempts + 1)` to a `while True` and this test never returns.

### Enterprise exploration

- The shelf is a dict here. Where does it live so four workers agree, and who reads it?
- A parked record is an unreported adverse event until somebody works it. What is the regulatory
  exposure of a week on the shelf?
- Feedback carries the validator's message into a prompt. What stops it carrying a subject name with
  it, and what does that cost at a thousand narratives an hour?

### Key takeaways

- A bound makes a loop finite. Only a diagnosis makes the later attempts worth paying for.
- Feed back the validator's own words. They name the field and the rule.
- Some errors must never be sent back. A model asked to fix an impossible value will invent one.
- Prove the bound with a test that hands the loop a model which can never succeed.